# 循环稳定性准入只读复核

当前正式准入0；8条只缺电流质量口径，另2条还涉及窗口或前作。七输入、科学标准和输出不变。数值覆盖不等于正式样本数。

## 范围与方法

仅只读查询本目录SQLite及核验文件，不下载、不写数据库、不训练。来源、审查出处及取得的新SI见主报告和external_sources。精确基线圈号未知保持未知；同文数据同组；可选孔体积疑点屏蔽。没有把不同DOI当成绝对独立的证明。


In [1]:
from pathlib import Path
import sqlite3, json
root = Path.cwd()
if not (root / 'cycling_v0_3r3.sqlite').exists():
    root = Path('D:\\机器学习-超级电容器\\循环稳定性数据库_v0.3_数据库优化_20261017\\第三轮_逐条准入闭环_20261017')
connection = sqlite3.connect((root / 'cycling_v0_3r3.sqlite').as_uri() + '?mode=ro', uri=True)
connection.row_factory = sqlite3.Row
print('正式候选终点:', connection.execute('SELECT count(*) FROM cycling_endpoints').fetchone()[0])
print('正式准入:', connection.execute('SELECT count(*) FROM training_ready').fetchone()[0])


正式候选终点: 89
正式准入: 0


## 主候选检查结果

分母已核和数值非空是两项不同检查。

In [2]:
flags = ['source_checked', 'material_checked', 'protocol_checked', 'baseline_checked', 'mass_basis_checked', 'independence_checked', 'freeze_cleared']
for flag in flags:
    passed = connection.execute(f'SELECT sum({flag}), count(*) FROM admissions WHERE record_id IN (SELECT record_id FROM primary_screening_candidates)').fetchone()
    print(f'{flag}: {passed[0]}/{passed[1]}')


source_checked: 10/10
material_checked: 10/10
protocol_checked: 9/10
baseline_checked: 10/10
mass_basis_checked: 0/10
independence_checked: 9/10
freeze_cleared: 9/10


## 逐条剩余阻碍

完整字段及出处见exports和mass_definition_audits。

In [3]:
query = """SELECT e.record_id,d.primary_disposition FROM endpoint_effective_review e
JOIN admission_decisions_r3 d USING(record_id)
WHERE e.record_id IN (SELECT record_id FROM primary_screening_candidates) ORDER BY e.record_id"""
for row in connection.execute(query):
    print(row['record_id'], row['primary_disposition'])


R091_CY01 blocked_current_definition_only
R093_CY01 blocked_current_definition_only
R103_CY01 blocked_current_definition_only
R104_CY01 blocked_current_definition_only
R107_CY01 blocked_current_definition_only
R107_CY02 blocked_current_definition_and_endpoint_window
R111_CY01 blocked_current_definition_only
SRC035_CY01 blocked_current_definition_only
SRC123_CY02 blocked_current_definition_and_predecessor_overlap
SRC124_CY01 blocked_current_definition_only


## 核验与限制

下面是数据完整性和工程回归结果，不是预测精度。

In [4]:
assert connection.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
assert not connection.execute('PRAGMA foreign_key_check').fetchall()
verification = json.loads((root / '核验结果_r3.json').read_text('utf-8'))
print('核验通过:', verification['passed'], '失败:', verification['failed'])
print('真实研究拟合:', verification['research_fits'])
for field in ['BET_m2_g', 'total_pore_volume_cm3_g']:
    counts = connection.execute(f'SELECT count(*),count(DISTINCT material_id),count(DISTINCT paper_id) FROM endpoint_effective_review WHERE {field} IS NOT NULL').fetchone()
    print(field, '字段覆盖(终点/材料/论文):', tuple(counts))
connection.close()


核验通过: 48 失败: 0
真实研究拟合: 0
BET_m2_g 字段覆盖(终点/材料/论文): (49, 37, 30)
total_pore_volume_cm3_g 字段覆盖(终点/材料/论文): (35, 27, 21)


## 结论与证据入口

现有材料已支持更多字段和审核决定，但无法替作者定义未明确的循环电流分母；正式准入仍为0。需要取得可引用的归一化说明，不能靠调参或统一除2解除。

[逐条报告](逐条准入结果与剩余证据.md)；[待补信息](仅剩关键缺证_待补信息.csv)；[核验结果](核验结果_r3.json)。本文件四个只读Python单元按顺序实际执行并保存输出；未使用Jupyter内核执行器，未安装任何环境依赖。
